# Benchmark Construction Pipeline

TODO: 
- include ORG-level insights and questions
- add contextual noise in review text, make the language more realistic
- gold answers need to be natural language

In [1]:
import json, math, re, random
import pandas as pd, numpy as np
from collections import defaultdict
from scipy.stats import norm
from dotenv import load_dotenv
from openai import OpenAI
import time
from pathlib import Path
import itertools
from tools import filter_data, exclude, count_by, sentiment_breakdown, add_shares, apply_min_volume, rank_top, share_of, two_prop_test, chi_squared, sample_reviews

pd.set_option("display.max_colwidth", 80)
random.seed(29)

In [3]:
OUT_DIR = Path("../benchmark_outputs") 
OUT_DIR.mkdir(parents=True, exist_ok=True)

FABSA = pd.read_csv("clean_fabsa.csv")
SENTS = ["positive", "negative", "neutral"]

In [2]:
# llm setup
load_dotenv()
client = OpenAI()
MODEL = "vertex_ai/gemini-2.5-flash" 

In [3]:
# cost logging
PRICE_IN, PRICE_OUT = 0.30/1e6, 2.50/1e6     # USD/token — verify current gemini-flash pricing
USAGE = defaultdict(lambda: {"calls":0, "in":0, "out":0})

def gemini(prompt, block, max_tokens=2000):
    r = client.chat.completions.create(model=MODEL,
            messages=[{"role":"user","content":prompt}], max_tokens=max_tokens)
    u = r.usage
    USAGE[block]["calls"]+=1; USAGE[block]["in"]+=u.prompt_tokens; USAGE[block]["out"]+=u.completion_tokens
    return r.choices[0].message.content

# helper - call gemini, strip json fences, parse JSON
def gemini_json(prompt, block, max_tokens=3000, retries=3):
    for attempt in range(retries):
        txt = gemini(prompt, block, max_tokens).strip()
        txt = re.sub(r"^```json|^```|```$", "", txt, flags=re.M).strip()
        try:
            return json.loads(txt)
        except json.JSONDecodeError:
            if attempt < retries - 1:
                continue
            print(f"[WARN] JSON parse failed after {retries} attempts. Raw:\n{txt[:200]}")
            raise

def cost_report():
    tot=0
    for b,d in USAGE.items():
        c=d["in"]*PRICE_IN+d["out"]*PRICE_OUT; tot+=c
        print(f"{b:12} calls={d['calls']:3d}  in={d['in']:7d}  out={d['out']:7d}  ${c:.4f}")
    print(f"{'TOTAL':12} {'':22} ${tot:.4f}")

# Step 1: plant insights by defining groups

In [4]:
# define aspect hierarchy
FABSA_INDUSTRIES = sorted(FABSA.industry.unique())

HIERARCHY = {
    "account-management": ["account-access"],
    "company-brand":      ["competitor", "general-satisfaction", "reviews"],
    "logistics-ride":     ["speed"],
    "online-experience":  ["app-website"],
    "booking-experience": ["ease-of-use"],
    "staff-support":      ["attitude-of-staff", "phone", "email"],
    "value":              ["discounts-promotions", "price-value-for-money"],
}

In [5]:
# sample 3 industries and 5 aspects

def sample_aspects(target=5):
    """Pick parents one by one, take all their children, stop when we have enough."""
    parents = list(HIERARCHY.keys())
    random.shuffle(parents)
    picked_parents, children = [], []
    for p in parents:
        picked_parents.append(p)
        children += HIERARCHY[p]
        if len(children) >= target:
            children = children[:target]
            break
    return picked_parents, children

INDUSTRIES = random.sample(FABSA_INDUSTRIES, 3)
PARENTS_USED, ASPECTS = sample_aspects(5)

print("industries:", INDUSTRIES)
print("parents:", PARENTS_USED, "-> aspects:", ASPECTS)

industries: ['Consulting', 'Banking', 'Information Technology']
parents: ['booking-experience', 'online-experience', 'logistics-ride', 'account-management', 'staff-support'] -> aspects: ['ease-of-use', 'app-website', 'speed', 'account-access', 'attitude-of-staff']


In [6]:
# plant sentiment splits per group
MAX_REVIEWS = 600
MIN_PER_GROUP = 35

GROUPS, gid = [], 1
raw_sizes = []
for ind in INDUSTRIES:
    for asp in ASPECTS:
        pos = round(random.uniform(0.20, 0.85), 2)
        neu = round(random.uniform(0.02, 0.06), 2)
        neg = round(1 - pos - neu, 4)
        raw_sizes.append((ind, asp, {"positive": pos, "negative": neg, "neutral": neu},
                          random.randint(45, 65)))

# guarantee minimum, distribute remaining budget proportionally
floor_total = MIN_PER_GROUP * len(raw_sizes)
remaining = MAX_REVIEWS - floor_total
raw_total = sum(r[3] for r in raw_sizes)
for ind, asp, shares, raw_n in raw_sizes:
    bonus = int(round(raw_n / raw_total * remaining)) if remaining > 0 else 0
    GROUPS.append((f"G{gid:04d}", ind, asp, shares, MIN_PER_GROUP + bonus))
    gid += 1
GRP = {g[0]: g for g in GROUPS}

print(f"budget: {MAX_REVIEWS} | actual: {sum(g[4] for g in GROUPS)} | min group: {min(g[4] for g in GROUPS)}")
for g in GROUPS[:3]:
    print(f"{g[0]}  {g[1]:25s}  {g[2]:25s}  pos={g[3]['positive']:.0%}  neg={g[3]['negative']:.0%}  neu={g[3]['neutral']:.0%}  n={g[4]}")

budget: 600 | actual: 599 | min group: 39
G0001  Consulting                 ease-of-use                pos=58%  neg=40%  neu=2%  n=39
G0002  Consulting                 app-website                pos=34%  neg=62%  neu=4%  n=39
G0003  Consulting                 speed                      pos=56%  neg=39%  neu=5%  n=41


In [ ]:
# expand groups into cells (exact counts)
def expand(g):
    _, ind, asp, sh, n = g
    c = {s: int(round(sh.get(s, 0) * n)) for s in SENTS}
    r = n - sum(c.values())
    if r: c[max(c, key=c.get)] += r     
    return [(ind, asp, s, c[s], g[0]) for s in SENTS if c[s] > 0]

CELLS = [c for g in GROUPS for c in expand(g)]
print(len(GROUPS), "groups ->", len(CELLS), "cells ->", sum(c[3] for c in CELLS), "reviews to generate")

15 groups -> 45 cells -> 599 reviews to generate


# Step 2: generate reviews according to the pre-defined patterns

In [ ]:
def fabsa_seeds(ind, asp, sent, k=3):
    m = FABSA[(FABSA.industry==ind) & (FABSA.child_aspect==asp) & (FABSA.sentiment==sent)]
    return m.text.dropna().drop_duplicates().head(k).tolist()

# produce a group of reviews
def gen_reviews(ind, asp, sent, n, seeds):
    ex = "\n".join(f"- {s}" for s in seeds) or "- (none)"
    revs = gemini_json(
      f"Write {n} short customer reviews for a {ind} company, each expressing "
      f"{sent.upper()} sentiment about '{asp}'. Match the style of these real examples:\n{ex}\n"
      f"Rules: each review MUST be 5-30 words, anonymise any brand as 'ORG', vary the wording. "
      f"Return ONLY a JSON list of strings.", block="generate")
    return [r for r in revs if 5 <= len(r.split()) <= 30]

# send a group of reviews to a separate Gemini call to verify aspect and sentiment labels
def verify(reviews, asp, sent):
    return gemini_json(
      f"For each review answer true only if it expresses {sent.upper()} sentiment about "
      f"'{asp}', else false. Return ONLY a JSON list of booleans, same order.\n"
      f"{json.dumps(reviews)}", block="verify")

In [ ]:
# build the corpus (slow)
start = time.time()
rows, rid = [], 0
for i, (ind, asp, sent, n, gid) in enumerate(CELLS, 1):
    seeds, kept = fabsa_seeds(ind, asp, sent), []
    for _ in range(2):
        need = n - len(kept)
        if need <= 0: break
        revs = gen_reviews(ind, asp, sent, need + 2, seeds)
        ok = verify(revs, asp, sent)
        kept += [r for r, g in zip(revs, ok) if g][:need]
    for t in kept[:n]:
        rows.append((rid, ind, asp, sent, t, gid)); rid += 1
    print(f"[{i}/{len(CELLS)}] {gid} {sent}: {len(kept[:n])}/{n}")

CORPUS = pd.DataFrame(rows, columns=["review_id","industry","child_aspect","sentiment","text","group_id"])
CORPUS.to_csv(OUT_DIR / "corpus.csv", index=False)

elapsed = time.time() - start
print(f"\ncorpus: {len(CORPUS)} reviews in {elapsed/60:.1f} min")
cost_report()

# 9mins, $0.24

[1/45] G0001 positive: 22/22
[2/45] G0001 negative: 16/16
[3/45] G0001 neutral: 1/1
[4/45] G0002 positive: 13/13
[5/45] G0002 negative: 24/24
[6/45] G0002 neutral: 2/2
[7/45] G0003 positive: 23/23
[8/45] G0003 negative: 16/16
[9/45] G0003 neutral: 2/2
[10/45] G0004 positive: 19/19
[11/45] G0004 negative: 19/19
[12/45] G0004 neutral: 2/2
[13/45] G0005 positive: 29/29
[14/45] G0005 negative: 10/10
[15/45] G0005 neutral: 1/1
[16/45] G0006 positive: 26/26
[17/45] G0006 negative: 12/12
[18/45] G0006 neutral: 1/1
[19/45] G0007 positive: 13/13
[20/45] G0007 negative: 24/24
[21/45] G0007 neutral: 2/2
[22/45] G0008 positive: 10/10
[23/45] G0008 negative: 29/29
[24/45] G0008 neutral: 1/1
[25/45] G0009 positive: 24/24
[26/45] G0009 negative: 14/14
[27/45] G0009 neutral: 2/2
[28/45] G0010 positive: 22/22
[29/45] G0010 negative: 16/16
[30/45] G0010 neutral: 2/2
[31/45] G0011 positive: 10/10
[32/45] G0011 negative: 30/30
[33/45] G0011 neutral: 1/1
[34/45] G0012 positive: 25/25
[35/45] G0012 negative

In [12]:
CORPUS.head()

,review_id,industry,child_aspect,sentiment,text,group_id
0,0,Consulting,ease-of-use,positive,ORG made the entire process incredibly simple. Highly recommend!,G0001
1,1,Consulting,ease-of-use,positive,Getting started with ORG was surprisingly easy. Great service!,G0001
2,2,Consulting,ease-of-use,positive,Their platform is so user-friendly. A joy to work with.,G0001
3,3,Consulting,ease-of-use,positive,No complications at all. ORG's system is very intuitive.,G0001
4,4,Consulting,ease-of-use,positive,Everything was explained clearly and easy to understand.,G0001


# Step 3: create task specs to derive answers and tool paths

In [18]:
# build solver functions for each task
# each solver function runs the tools on the corpus to produce the gold answer and records the tool calls

# ---- 1-step solvers (descriptive) ----

def solve_count(ind, asp, sent):
    s = filter_data(CORPUS, industry=ind, child_aspect=asp, sentiment=sent)
    return len(s), [
        {"tool":"count","args":{"industry":ind,"child_aspect":asp,"sentiment":sent}}]

def solve_share(ind, asp, sent):
    r = share_of(filter_data(CORPUS, industry=ind, child_aspect=asp), sent)
    return r["share"], [
        {"tool":"share_of","args":{"industry":ind,"child_aspect":asp,"sentiment":sent}}]

# ---- 2-step solvers (descriptive or diagnostic) ----

def solve_top_k(ind, k, by):
    filt = {"industry": ind, **({"sentiment":"negative"} if by=="negative" else {})}
    top = rank_top(count_by(filter_data(CORPUS, **filt), "child_aspect"), by="count", top_n=k)
    return top.child_aspect.tolist(), [
        {"tool":"count_by","args":{**filt,"group_by":"child_aspect"}},
        {"tool":"rank_top","args":{"by":"count","top_n":k}}]

def solve_split(ind):
    sh = add_shares(sentiment_breakdown(filter_data(CORPUS, industry=ind), "industry"))
    v = {k: float(sh[k].iloc[0]) for k in ["pos_share","neg_share","neu_share"]}
    return v, [
        {"tool":"sentiment_breakdown","args":{"industry":ind,"group_by":"industry"}},
        {"tool":"add_shares","args":{}}]

def solve_compare(ind_a, ind_b, asp, sent):
    ra = share_of(filter_data(CORPUS, industry=ind_a, child_aspect=asp), sent)
    rb = share_of(filter_data(CORPUS, industry=ind_b, child_aspect=asp), sent)
    return {"a":ra["share"],"b":rb["share"],"higher":"a" if (ra["share"] or 0)>(rb["share"] or 0) else "b"}, [
        {"tool":"share_of","args":{"industry":ind_a,"child_aspect":asp,"sentiment":sent}},
        {"tool":"share_of","args":{"industry":ind_b,"child_aspect":asp,"sentiment":sent}}]

# ---- 3-step solver (diagnostic) ----

def solve_two_prop(ind_a, ind_b, asp, sent):
    da = filter_data(CORPUS, industry=ind_a, child_aspect=asp)
    db = filter_data(CORPUS, industry=ind_b, child_aspect=asp)
    r = two_prop_test(da, db, sent)
    return r, [
        {"tool":"share_of","args":{"industry":ind_a,"child_aspect":asp,"sentiment":sent}},
        {"tool":"share_of","args":{"industry":ind_b,"child_aspect":asp,"sentiment":sent}},
        {"tool":"two_prop_test","args":{"sentiment":sent}}]

# ---- 4-step solver (diagnostic) ----

def solve_driver(ind, k):
    sh = add_shares(sentiment_breakdown(filter_data(CORPUS, industry=ind), "child_aspect"))
    vol = apply_min_volume(sh, min_volume=5)
    top = rank_top(vol, by="neg_share", top_n=k)
    v = list(zip(top.child_aspect, top.neg_share.round(4)))
    return v, [
        {"tool":"sentiment_breakdown","args":{"industry":ind,"group_by":"child_aspect"}},
        {"tool":"add_shares","args":{}},
        {"tool":"apply_min_volume","args":{"min_volume":5}},
        {"tool":"rank_top","args":{"by":"neg_share","top_n":k}}]

SOLVERS = {
    "solve_count": solve_count, "solve_share": solve_share,
    "solve_top_k": solve_top_k, "solve_split": solve_split,
    "solve_compare": solve_compare, "solve_two_prop": solve_two_prop,
    "solve_driver": solve_driver,
} 

Task structure:
- tid: task ID, numbered by increasing difficulty
- type: question intent (descriptive, diagnostic or prescriptive)
- inds: industries
- solver: function that computes the gold answer 
- args: arguments to pass to that solver
- spec: structured description of the task, input to Gemini to write the question
- steps: number of tool calls in the gold path

In [20]:
# generate task specs. Sample parameters and solvers at increasing difficulty level
def auto_tasks(industries, aspects, seed=42):
    rng = random.Random(seed)
    T = []
    used_aspects = set()   # prevent duplicate aspect picks in diagnostic

    def _add(tid, type, inds, solver, args, spec, steps):
        T.append((tid, type, set(inds), solver, args, spec, steps))

    # ---- DESCRIPTIVE (7 tasks) ----
    # 1-step (count or share on one industry+aspect)
    #   Shuffled (industry, aspect) pairs; alternating count/share and pos/neg
    pool = [(ind, asp) for ind in industries for asp in aspects]
    rng.shuffle(pool)
    ops = [("count","negative"), ("share","positive"), ("share","negative"), ("share","positive")]
    for i, ((ind, asp), (op, sent)) in enumerate(zip(pool, ops), 1):
        _add(f"DESC-{i:02d}", "descriptive", [ind],
             f"solve_{op}", (ind, asp, sent),
             {"op": op, "ind": ind, "aspect": asp, "sent": sent}, steps=1)

    # 2-step (one per industry, shuffled type)
    #   top-1 by volume, top-3 by complaints, or overall sentiment split
    types = [("top_k","volume",1), ("top_k","negative",3), ("split",None,None)]
    rng.shuffle(types)
    for i, (ind, (typ, by, k)) in enumerate(zip(industries, types), 5):
        if typ == "top_k":
            _add(f"DESC-{i:02d}", "descriptive", [ind],
                 "solve_top_k", (ind, k, by),
                 {"op":f"top_{by}", "ind":ind, **({"k":k} if k and k>1 else {})}, steps=2)
        else:
            _add(f"DESC-{i:02d}", "descriptive", [ind],
                 "solve_split", (ind,),
                 {"op":"split", "ind":ind}, steps=2)

    # ---- DIAGNOSTIC (5 tasks) ----
    pairs = list(itertools.combinations(industries, 2))

    # 2-step (compare shares across 2 industries, no formal test)
    pa, pb = rng.choice(pairs)
    asp = rng.choice(aspects); used_aspects.add(asp)
    sent = rng.choice(["positive","negative"])
    _add("DIAG-01", "diagnostic", [pa, pb],
         "solve_compare", (pa, pb, asp, sent),
         {"op":"compare","a":pa,"b":pb,"aspect":asp,"sent":sent}, steps=2)

    # 3-step (two-proportion z-test on remaining pairs)
    remaining = [p for p in pairs if set(p) != {pa, pb}]
    for i, (pa2, pb2) in enumerate(remaining, 2):
        avail = [a for a in aspects if a not in used_aspects]
        if not avail: avail = aspects
        asp = rng.choice(avail); used_aspects.add(asp)
        sent = rng.choice(["positive","negative"])
        _add(f"DIAG-{i:02d}", "diagnostic", [pa2, pb2],
             "solve_two_prop", (pa2, pb2, asp, sent),
             {"op":"test","a":pa2,"b":pb2,"aspect":asp,"sent":sent}, steps=3)

    # 4-step (rank negative-sentiment drivers in one industry)
    dr_inds = rng.sample(industries, 2)
    for i, (ind, k) in enumerate(zip(dr_inds, [1, 2]), 4):
        _add(f"DIAG-{i:02d}", "diagnostic", [ind],
             "solve_driver", (ind, k),
             {"op":"driver","ind":ind,"k":k}, steps=4)

    # ---- PRESCRIPTIVE (3 tasks) ----
    # Each: ≥1 descriptive sub-task + ≥1 diagnostic sub-task + 1 summarise
    for i, ind in enumerate(industries, 1):
        desc = [t for t in T if t[1]=="descriptive" and t[2]=={ind}]
        diag = [t for t in T if t[1]=="diagnostic" and ind in t[2]]
        pick_d = desc[:1]      # 1 descriptive sub-task
        pick_g = diag[:1]      # 1 diagnostic sub-task
        subs = [t[0] for t in pick_d + pick_g]
        sub_steps = sum(t[6] for t in pick_d + pick_g)
        _add(f"PRES-{i:02d}", "prescriptive", [ind],
             "solve_compose", (subs,),
             {"op":"recommend","ind":ind,"subs":subs}, steps=sub_steps + 1)

    return T

T = auto_tasks(INDUSTRIES, ASPECTS, seed=42)
print(f"{len(T)} tasks\n")
for tid, type, inds, solver, args, spec, steps in T:
    print(f"{tid}  {steps}-step  {type:12s}  {spec}")

15 tasks

DESC-01  1-step  descriptive   {'op': 'count', 'ind': 'Banking', 'aspect': 'account-access', 'sent': 'negative'}
DESC-02  1-step  descriptive   {'op': 'share', 'ind': 'Information Technology', 'aspect': 'account-access', 'sent': 'positive'}
DESC-03  1-step  descriptive   {'op': 'share', 'ind': 'Banking', 'aspect': 'speed', 'sent': 'negative'}
DESC-04  1-step  descriptive   {'op': 'share', 'ind': 'Banking', 'aspect': 'app-website', 'sent': 'positive'}
DESC-05  2-step  descriptive   {'op': 'top_negative', 'ind': 'Consulting', 'k': 3}
DESC-06  2-step  descriptive   {'op': 'split', 'ind': 'Banking'}
DESC-07  2-step  descriptive   {'op': 'top_volume', 'ind': 'Information Technology'}
DIAG-01  2-step  diagnostic    {'op': 'compare', 'a': 'Consulting', 'b': 'Banking', 'aspect': 'app-website', 'sent': 'positive'}
DIAG-02  3-step  diagnostic    {'op': 'test', 'a': 'Consulting', 'b': 'Information Technology', 'aspect': 'ease-of-use', 'sent': 'positive'}
DIAG-03  3-step  diagnostic    {

In [21]:
# run solvers to get gold answers and paths (descriptive and diagnostic)
records = {}
for tid, type, inds, solver_name, args, spec, steps in T:
    if solver_name == "solve_compose":
        continue
    fn = SOLVERS[solver_name]
    ans, path = fn(*args)
    assert len(path) == steps, f"{tid}: expected {steps} steps, got {len(path)}"
    records[tid] = dict(task_id=tid, type=type, steps=steps,
                        industries=sorted(inds), spec=spec,
                        gold_answer=ans, gold_tool_path=path)

for tid in sorted(records)[:5]:
    print(f"{tid} ({records[tid]['steps']}-step): {records[tid]['gold_answer']}")

DESC-01 (1-step): 14
DESC-02 (1-step): 0.7
DESC-03 (1-step): 0.725
DESC-04 (1-step): 0.333
DESC-05 (2-step): ['app-website', 'account-access', 'ease-of-use']


In [22]:
# add prescriptive tasks
for tid, type, inds, solver_name, args, spec, steps in T:
    if solver_name != "solve_compose":
        continue
    subs = spec["subs"]
    flags = {s: records[s]["gold_answer"] for s in subs}
    path = ([step for s in subs for step in records[s]["gold_tool_path"]]
            + [{"tool": "summarise", "args": {"from": subs}}])
    assert len(path) == steps, f"{tid}: expected {steps} steps, got {len(path)}"
    records[tid] = dict(task_id=tid, type=type, steps=steps,
                        industries=sorted(inds), spec=spec,
                        gold_answer={"findings": flags},
                        gold_tool_path=path)
    print(f"{tid} ({steps}-step): composes {subs} -> {len(path)} steps in path")

PRES-01 (5-step): composes ['DESC-05', 'DIAG-01'] -> 5 steps in path
PRES-02 (4-step): composes ['DESC-01', 'DIAG-01'] -> 4 steps in path
PRES-03 (5-step): composes ['DESC-02', 'DIAG-02'] -> 5 steps in path


# Step 4: generate task questions

In [24]:
# use LLM to write questions from task specs
start = time.time()
def brief(sp):
    o = sp["op"]
    if o == "count":        return f"How many complaints {sp['ind']} received about {sp['aspect']}."
    if o == "share":        return f"The share of {sp['ind']} feedback on {sp['aspect']} that is {'praise' if sp['sent']=='positive' else 'complaints'}."
    if o == "top_volume":   return f"Which single topic {sp['ind']} customers mention most."
    if o == "top_negative": return f"The {sp['k']} topics with the most complaints in {sp['ind']}."
    if o == "split":        return f"The overall breakdown of happy vs unhappy {sp['ind']} customers."
    if o == "compare":      return f"Whether {sp['a']} or {sp['b']} customers have more {'complaints about' if sp['sent']=='negative' else 'praise for'} {sp['aspect']}."
    if o == "test":         return f"Whether the difference in {'complaints about' if sp['sent']=='negative' else 'praise for'} {sp['aspect']} between {sp['a']} and {sp['b']} is statistically significant."
    if o == "driver":       return (f"The single biggest driver of dissatisfaction in {sp['ind']}." if sp['k']==1
                                    else f"The top {sp['k']} drivers of dissatisfaction in {sp['ind']}.")
    if o == "recommend":    return f"Diagnose the biggest customer problems in {sp['ind']} and recommend the top fixes."

briefs = {tid: brief(r["spec"]) for tid, r in records.items()}

qs = gemini_json(
  "You are a CX analytics lead. For each item, write ONE specific, realistic, HARD business "
  "question a stakeholder would ask, whose answer requires exactly the analysis described.\n"
  "Rules: natural business language only — say 'complaints', 'issues', 'happy', 'like', 'praise', "
  "'frustrated'; NEVER use words like 'sentiment', 'aspect', 'positive/negative class', or column "
  "names. Translate topics into plain phrasing (e.g. 'app-website'->'the app or website', "
  "'ease-of-use'->'how easy the service is to use', 'account-access'->'signing in or accessing "
  "their account'). Keep the exact industry names and any counts (e.g. 'top 3'). "
  "Return ONLY a JSON object mapping id -> question string.\n"
  + json.dumps(briefs), block="questions", max_tokens=3000)

for tid, q in qs.items():
    records[tid]["question"] = q
    print(f"{tid}: {q}")

elapsed = time.time() - start
print(f"\nquestions generated in {elapsed:.1f}s")
cost_report()

DESC-01: How many customers in Banking are complaining about signing in or accessing their account?
DESC-02: What proportion of feedback from Information Technology customers about signing in or accessing their account is praise?
DESC-03: What proportion of feedback from Banking customers regarding speed is complaints?
DESC-04: What proportion of feedback from Banking customers about their experience with the app or website is praise?
DESC-05: What are the top 3 things Consulting customers complain about the most?
DESC-06: Can we see the overall breakdown of how happy or unhappy our Banking customers are?
DESC-07: What's the one thing Information Technology customers talk about more than anything else?
DIAG-01: Are customers in Consulting or Banking more likely to praise their experience with the app or website?
DIAG-02: Is the difference in how much Consulting customers praise how easy the service is to use compared to Information Technology customers a real, meaningful difference, or

In [ ]:
# save everything and print cost report
def _clean(r):
    """Make record JSON-serialisable (sets -> sorted lists)."""
    r = dict(r)
    if isinstance(r.get("industries"), set):
        r["industries"] = sorted(r["industries"])
    return r

order = [t[0] for t in T] 

with open(OUT_DIR / "tasks.jsonl", "w") as f:
    for tid in order:
        f.write(json.dumps(_clean(records[tid])) + "\n")

# print(f"wrote {OUT_DIR / 'tasks.jsonl'} ({len(records)} tasks)")
tasks_df = pd.DataFrame([_clean(records[tid]) for tid in order])
tasks_df.to_csv(OUT_DIR / "tasks.csv", index=False)
print(f"wrote {OUT_DIR / 'tasks.csv'}")
print(f"wrote {OUT_DIR / 'corpus.csv'} ({len(CORPUS)} reviews)\n")
cost_report()

wrote ..\benchmark_outputs\tasks.csv
wrote ..\benchmark_outputs\corpus.csv (599 reviews)

generate     calls= 46  in=   4568  out=  57456  $0.1450
verify       calls= 46  in=  12568  out=  39196  $0.1018
questions    calls=  2  in=    940  out=   4572  $0.0117
TOTAL                               $0.2585


# Step 5: verify gold tool paths

Sanity check: independently **re-run each recorded `gold_tool_path`** against `CORPUS`
(calling the real tool functions directly, *not* the solver functions that produced them) and
confirm it reproduces `gold_answer`. A wrong tool name or argument in a path would make the two
diverge. The final answer is read off the last tool's output, shaped per the task's `spec["op"]`.

In [4]:
# self-contained: load the saved corpus + tasks.csv, then re-run every recorded
# gold_tool_path against CORPUS with the real tools and check it reproduces gold_answer.
# NOTE: "count" and "summarise" are abstract tools in the path (no standalone fn):
#   count = filter_data(...) + len      summarise = compose sub-task answers
import ast

CORPUS = pd.read_csv(OUT_DIR / "corpus.csv")

# tasks.csv stores dict/list columns as Python repr strings -> ast.literal_eval them.
# duplicate task_ids (e.g. PRES rows written twice) collapse to one entry, keyed by id.
_tasks = pd.read_csv(OUT_DIR / "tasks.csv")
records = {}
for _, r in _tasks.iterrows():
    records[r["task_id"]] = {
        "task_id": r["task_id"],
        "steps": int(r["steps"]),
        "spec": ast.literal_eval(r["spec"]),
        "gold_answer": ast.literal_eval(str(r["gold_answer"])),
        "gold_tool_path": ast.literal_eval(r["gold_tool_path"]),
        "question": r["question"],
    }

_SLICE_KEYS = ("industry", "data_source", "parent_aspect", "child_aspect", "sentiment")

def _slice(args, keys=_SLICE_KEYS):
    return filter_data(CORPUS, **{k: args[k] for k in keys if k in args})

def run_gold_path(path, spec, records):
    op = spec["op"]
    cur = None                            # current DataFrame for chained ops
    share_slices, share_vals = [], []     # from share_of steps -> feed compare / two_prop_test
    scalar = tp = None
    for step in path:
        t, a = step["tool"], step["args"]
        if t == "count":                                  # abstract: filter + count rows
            scalar = len(_slice(a))
        elif t == "share_of":
            df = _slice(a, ("industry", "child_aspect"))
            share_slices.append(df)
            share_vals.append(share_of(df, a["sentiment"])["share"])
        elif t == "count_by":
            cur = count_by(_slice(a, ("industry", "sentiment", "child_aspect", "parent_aspect")), a["group_by"])
        elif t == "sentiment_breakdown":
            cur = sentiment_breakdown(_slice(a, ("industry", "child_aspect")), a["group_by"])
        elif t == "add_shares":
            cur = add_shares(cur)
        elif t == "apply_min_volume":
            cur = apply_min_volume(cur, min_volume=a.get("min_volume", 30))
        elif t == "rank_top":
            cur = rank_top(cur, by=a["by"], top_n=a["top_n"])
        elif t == "two_prop_test":
            tp = two_prop_test(share_slices[-2], share_slices[-1], a["sentiment"])
        elif t == "summarise":
            pass                                          # composition handled in read-off
        else:
            raise ValueError(f"unknown tool in path: {t}")

    # read the final answer off the last tool's output, shaped per task op
    if op == "count":
        return scalar
    if op == "share":
        return share_vals[-1]
    if op in ("top_volume", "top_negative"):
        return cur["child_aspect"].tolist()
    if op == "split":
        return {k: float(cur[k].iloc[0]) for k in ("pos_share", "neg_share", "neu_share")}
    if op == "compare":
        a_, b_ = share_vals[0], share_vals[1]
        return {"a": a_, "b": b_, "higher": "a" if (a_ or 0) > (b_ or 0) else "b"}
    if op == "test":
        return tp
    if op == "driver":
        return [[asp, float(round(ns, 4))] for asp, ns in zip(cur["child_aspect"], cur["neg_share"])]
    if op == "recommend":
        return {"findings": {s: run_gold_path(records[s]["gold_tool_path"], records[s]["spec"], records)
                             for s in spec["subs"]}}
    raise ValueError(f"unknown op: {op}")


def _match(a, b, tol=1e-4):
    """Tolerant structural comparison: list==tuple, floats within tol, exact otherwise."""
    if isinstance(a, bool) or isinstance(b, bool):
        return a == b
    if a is None or b is None:
        return a is None and b is None
    if isinstance(a, (int, float)) and isinstance(b, (int, float)):
        return math.isclose(float(a), float(b), abs_tol=tol)
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_match(a[k], b[k], tol) for k in a)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_match(x, y, tol) for x, y in zip(a, b))
    return a == b


print(f"Verifying {len(records)} gold tool paths from tasks.csv against CORPUS ({len(CORPUS)} reviews)")
print("=" * 78)
n_pass = 0
for tid, rec in records.items():
    got = run_gold_path(rec["gold_tool_path"], rec["spec"], records)
    ok = _match(got, rec["gold_answer"])
    n_pass += ok
    print(f"\n[{tid}] {rec['steps']}-step  {rec['question']}")
    print(f"  gold answer : {rec['gold_answer']}")
    print(f"  path re-ran : {got}")
    print(f"  {'PASS' if ok else 'FAIL <<<<<<'}")

print("\n" + "=" * 78)
print(f"{n_pass}/{len(records)} tasks: gold_tool_path reproduces gold_answer")

Verifying 15 gold tool paths from tasks.csv against CORPUS (599 reviews)

[DESC-01] 1-step  How many customers in Banking are complaining about signing in or accessing their account?
  gold answer : 14
  path re-ran : 14
  PASS

[DESC-02] 1-step  What proportion of feedback from Information Technology customers about signing in or accessing their account is praise?
  gold answer : 0.7
  path re-ran : 0.7
  PASS

[DESC-03] 1-step  What proportion of feedback from Banking customers regarding speed is complaints?
  gold answer : 0.725
  path re-ran : 0.725
  PASS

[DESC-04] 1-step  What proportion of feedback from Banking customers about their experience with the app or website is praise?
  gold answer : 0.333
  path re-ran : 0.333
  PASS

[DESC-05] 2-step  What are the top 3 things Consulting customers complain about the most?
  gold answer : ['app-website', 'account-access', 'ease-of-use']
  path re-ran : ['app-website', 'account-access', 'ease-of-use']
  PASS

[DESC-06] 2-step  Can we 